# Task Inspection and Leakage Audit

This notebook illustrates all registered OCEL-OCP tasks and motivates design choices for:

- `timedelta` (window and sampling spacing)
- `num_eval_timestamps`
- train/val/test timestamp boundaries
- leakage safety checks based on effective label horizons

It uses helpers from `scripts/inspection/utils` where applicable.

In [ ]:
from pathlib import Path
import inspect
import os
import re
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
ROOT = next(
    (p for p in candidates if (p / "data").exists() and (p / "task").exists()),
    cwd,
)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import data  # noqa: F401  # activates Table.load dtype normalization
import task  # noqa: F401  # registers custom task classes
from data.const import TIME_COL
from data.dataset import register_all_datasets
from relbench.base import TaskType
from relbench.datasets import get_dataset
from relbench.tasks import get_task, get_task_names
from scripts.inspection.utils import (
    configure_plot_style,
    event_object_recent_matrix,
    sample_window_graph_sizes,
    summarize_interval_stability,
    summarize_sampled_window_graph_sizes,
)

configure_plot_style()
register_all_datasets()
print(f"Working directory: {Path.cwd()}")

In [ ]:
DATASETS = ["order_management", "container_logistics", "bpi2017", "bpi2019"]

# Primary mode. If False, notebook will still retry failed loads with download=True.
ALLOW_DOWNLOAD = False
RETRY_DOWNLOAD_ON_FAILURE = True

WINDOW_GRAPH_NUM_ROOTS = 64
WINDOW_GRAPH_RANDOM_STATE = 0
STABILITY_BUCKET_HOURS = [1, 2, 3, 4, 6, 8, 12, 24]

In [ ]:
def _extract_class_rationale(task_obj) -> str:
    src = inspect.getsource(task_obj.__class__)
    comments = []
    for line in src.splitlines():
        stripped = line.strip()
        if stripped.startswith("#"):
            comments.append(re.sub(r"^#\s?", "", stripped))
    return " ".join(comments)


def _safe_timedelta_hours(td) -> float | None:
    if td is None:
        return None
    try:
        return float(pd.Timedelta(td).total_seconds() / 3600.0)
    except Exception:
        return None


def summarize_split(task_obj, split: str) -> dict[str, object]:
    table = task_obj.get_table(split)
    df = table.df
    time_col = task_obj.time_col
    if time_col in df.columns:
        ts = pd.to_datetime(df[time_col], errors="coerce").dropna()
        min_t = ts.min() if not ts.empty else pd.NaT
        max_t = ts.max() if not ts.empty else pd.NaT
        n_ts = int(ts.nunique())
    else:
        min_t = pd.NaT
        max_t = pd.NaT
        n_ts = 0
    return {
        "rows": int(len(df)),
        "unique_timestamps": n_ts,
        "min_time": min_t,
        "max_time": max_t,
        "has_target": bool(task_obj.target_col in df.columns),
    }


def summarize_target(task_obj, split: str = "train") -> dict[str, object]:
    df = task_obj.get_table(split).df
    if task_obj.target_col not in df.columns:
        return {"target_available": False}
    y = df[task_obj.target_col]
    out: dict[str, object] = {"target_available": True}
    if task_obj.task_type == TaskType.BINARY_CLASSIFICATION:
        y_num = pd.to_numeric(y.astype("int64"), errors="coerce").dropna()
        out["positive_rate"] = float(y_num.mean()) if not y_num.empty else float("nan")
    elif task_obj.task_type == TaskType.REGRESSION:
        y_num = pd.to_numeric(y, errors="coerce").dropna()
        y_hours = y_num / 3600.0
        out["p50_hours"] = float(y_hours.quantile(0.50)) if not y_hours.empty else float("nan")
        out["p90_hours"] = float(y_hours.quantile(0.90)) if not y_hours.empty else float("nan")
        out["mean_hours"] = float(y_hours.mean()) if not y_hours.empty else float("nan")
    return out


def _probe_task_materialization(task_obj) -> str | None:
    """Return an error string if core split tables cannot be materialized."""
    try:
        task_obj.get_table("train")
        task_obj.get_table("val")
        return None
    except Exception as exc:
        return str(exc)


def _get_task_with_retry(dataset_name: str, task_name: str, download: bool, retry_download: bool):
    try:
        task_obj = get_task(dataset_name, task_name, download=download)
        probe_err = _probe_task_materialization(task_obj)
        if probe_err is None:
            return task_obj, None
        raise RuntimeError(probe_err)
    except Exception as exc:
        if retry_download and not download:
            try:
                task_obj = get_task(dataset_name, task_name, download=True)
                probe_err = _probe_task_materialization(task_obj)
                if probe_err is None:
                    return task_obj, None
                return None, f"{exc} | retry(download=True) probe failed: {probe_err}"
            except Exception as exc_retry:
                return None, f"{exc} | retry(download=True) failed: {exc_retry}"
        return None, str(exc)


def collect_task_bundle(dataset_name: str, download: bool = False, retry_download: bool = True) -> dict[str, object]:
    ds = None
    db = None
    ds_error = None
    try:
        ds = get_dataset(dataset_name, download=download)
        db = ds.get_db()
    except Exception as exc:
        if retry_download and not download:
            ds = get_dataset(dataset_name, download=True)
            db = ds.get_db()
        else:
            ds_error = str(exc)

    if ds is None or db is None:
        return {
            "dataset": None,
            "db": None,
            "task_names": [],
            "task_objs": {},
            "task_errors": {},
            "dataset_error": ds_error,
        }

    task_names = sorted(get_task_names(dataset_name))
    task_objs = {}
    task_errors = {}
    for task_name in task_names:
        task_obj, err = _get_task_with_retry(dataset_name, task_name, download, retry_download)
        if task_obj is None:
            task_errors[task_name] = err
        else:
            task_objs[task_name] = task_obj

    return {
        "dataset": ds,
        "db": db,
        "task_names": sorted(task_objs.keys()),
        "task_objs": task_objs,
        "task_errors": task_errors,
        "dataset_error": None,
    }


def _effective_horizon_seconds(task_name: str, task_obj, split: str) -> pd.Series:
    df = task_obj.get_table(split).df
    n = len(df)

    if task_obj.task_type == TaskType.REGRESSION and task_obj.target_col in df.columns:
        y = pd.to_numeric(df[task_obj.target_col], errors="coerce")
        return y.clip(lower=0)

    td = getattr(task_obj, "timedelta", None)
    if td is not None:
        sec = pd.Timedelta(td).total_seconds()
        return pd.Series(np.full(n, sec), index=df.index)

    return pd.Series(np.full(n, np.nan), index=df.index)


def leakage_check_against_boundary(task_name: str, task_obj, split: str, boundary_ts: pd.Timestamp) -> dict[str, object]:
    df = task_obj.get_table(split).df
    time_col = task_obj.time_col
    if time_col not in df.columns or len(df) == 0:
        return {
            "rows": int(len(df)),
            "rows_with_horizon": 0,
            "cross_boundary_rows": 0,
            "cross_boundary_rate": 0.0,
            "max_label_end": pd.NaT,
        }

    t = pd.to_datetime(df[time_col], errors="coerce")
    horizon_sec = _effective_horizon_seconds(task_name, task_obj, split)
    valid = t.notna() & horizon_sec.notna()
    label_end = t[valid] + pd.to_timedelta(horizon_sec[valid], unit="s")

    if label_end.empty:
        return {
            "rows": int(len(df)),
            "rows_with_horizon": 0,
            "cross_boundary_rows": 0,
            "cross_boundary_rate": 0.0,
            "max_label_end": pd.NaT,
        }

    cross = label_end > boundary_ts
    return {
        "rows": int(len(df)),
        "rows_with_horizon": int(valid.sum()),
        "cross_boundary_rows": int(cross.sum()),
        "cross_boundary_rate": float(cross.mean()),
        "max_label_end": label_end.max(),
    }


def spacing_check(task_obj, split: str) -> dict[str, object]:
    df = task_obj.get_table(split).df
    if task_obj.time_col not in df.columns:
        return {
            "unique_timestamps": 0,
            "violations": 0,
            "min_gap_hours": float("nan"),
            "required_gap_hours": _safe_timedelta_hours(getattr(task_obj, "timedelta", None)),
        }

    ts = pd.to_datetime(df[task_obj.time_col], errors="coerce").dropna().drop_duplicates().sort_values()
    if len(ts) < 2:
        return {
            "unique_timestamps": int(len(ts)),
            "violations": 0,
            "min_gap_hours": float("nan"),
            "required_gap_hours": _safe_timedelta_hours(getattr(task_obj, "timedelta", None)),
        }

    diffs = ts.diff().dropna()
    required = pd.Timedelta(getattr(task_obj, "timedelta", pd.Timedelta(0)))
    violations = int((diffs < required).sum())
    return {
        "unique_timestamps": int(len(ts)),
        "violations": violations,
        "min_gap_hours": float(diffs.min().total_seconds() / 3600.0),
        "required_gap_hours": _safe_timedelta_hours(required),
    }

In [ ]:
analysis: dict[str, dict[str, object]] = {}

for dataset_name in DATASETS:
    bundle = collect_task_bundle(
        dataset_name,
        download=ALLOW_DOWNLOAD,
        retry_download=RETRY_DOWNLOAD_ON_FAILURE,
    )
    analysis[dataset_name] = bundle

    if bundle["dataset_error"] is not None:
        print(f"Dataset failed: {dataset_name} -> {bundle['dataset_error']}")
        continue

    print(
        f"Loaded {dataset_name}: {len(bundle['task_names'])} / {len(get_task_names(dataset_name))} task(s)"
    )
    if bundle["task_errors"]:
        print(f"  Task load failures ({len(bundle['task_errors'])}):")
        for name, err in bundle["task_errors"].items():
            print(f"    - {name}: {err}")

In [ ]:
load_summary_rows = []
for dataset_name, bundle in analysis.items():
    load_summary_rows.append(
        {
            "dataset": dataset_name,
            "dataset_loaded": bundle["dataset_error"] is None,
            "loaded_tasks": len(bundle["task_names"]),
            "failed_tasks": len(bundle["task_errors"]),
            "dataset_error": bundle["dataset_error"],
        }
    )
load_summary_df = pd.DataFrame(load_summary_rows)
display(load_summary_df)

In [ ]:
task_error_rows = []
for dataset_name, bundle in analysis.items():
    for task_name, err in bundle["task_errors"].items():
        task_error_rows.append({"dataset": dataset_name, "task": task_name, "error": err})
task_error_df = pd.DataFrame(task_error_rows)
if task_error_df.empty:
    display(Markdown("No task materialization errors."))
else:
    display(task_error_df.sort_values(["dataset", "task"], kind="stable"))

## 1) Task Catalog (All Loaded Tasks)

In [ ]:
catalog_rows = []

for dataset_name, bundle in analysis.items():
    if bundle["dataset_error"] is not None:
        continue
    for task_name in bundle["task_names"]:
        task_obj = bundle["task_objs"][task_name]
        td = getattr(task_obj, "timedelta", None)
        catalog_rows.append(
            {
                "dataset": dataset_name,
                "task": task_name,
                "task_class": task_obj.__class__.__name__,
                "task_type": str(task_obj.task_type).split(".")[-1],
                "timedelta": str(td) if td is not None else None,
                "timedelta_hours": _safe_timedelta_hours(td),
                "num_eval_timestamps": int(getattr(task_obj, "num_eval_timestamps", 1)),
                "object_type": getattr(task_obj, "object_type", None),
                "positive_event_type": getattr(task_obj, "positive_event_type", None),
                "rationale_comments": _extract_class_rationale(task_obj),
            }
        )

catalog_df = pd.DataFrame(catalog_rows).sort_values(["dataset", "task"], kind="stable")
display(catalog_df)

## 2) Split Boundaries and Coverage

In [ ]:
split_rows = []

for dataset_name, bundle in analysis.items():
    if bundle["dataset_error"] is not None:
        continue
    ds = bundle["dataset"]
    for task_name in bundle["task_names"]:
        task_obj = bundle["task_objs"][task_name]
        train_info = summarize_split(task_obj, "train")
        val_info = summarize_split(task_obj, "val")
        test_info = summarize_split(task_obj, "test")
        split_rows.append(
            {
                "dataset": dataset_name,
                "task": task_name,
                "dataset_val_ts": ds.val_timestamp,
                "dataset_test_ts": ds.test_timestamp,
                "train_rows": train_info["rows"],
                "train_unique_ts": train_info["unique_timestamps"],
                "train_min": train_info["min_time"],
                "train_max": train_info["max_time"],
                "val_rows": val_info["rows"],
                "val_unique_ts": val_info["unique_timestamps"],
                "val_min": val_info["min_time"],
                "val_max": val_info["max_time"],
                "test_rows": test_info["rows"],
                "test_unique_ts": test_info["unique_timestamps"],
                "test_min": test_info["min_time"],
                "test_max": test_info["max_time"],
            }
        )

splits_df = pd.DataFrame(split_rows).sort_values(["dataset", "task"], kind="stable")
display(splits_df)

## 3) Leakage Audit

Two explicit checks:

1. **Spacing check**: split timestamps are at least `timedelta` apart.
2. **Boundary-horizon check**: `label_end = observation_time + effective_horizon` must not cross into the next split boundary.

Effective horizon used here:
- Regression: realized target horizon (`target` in seconds)
- Binary/event-window tasks: configured `timedelta`

In [ ]:
spacing_rows = []

for dataset_name, bundle in analysis.items():
    if bundle["dataset_error"] is not None:
        continue
    for task_name in bundle["task_names"]:
        task_obj = bundle["task_objs"][task_name]
        for split in ["train", "val", "test"]:
            chk = spacing_check(task_obj, split)
            spacing_rows.append(
                {
                    "dataset": dataset_name,
                    "task": task_name,
                    "split": split,
                    **chk,
                }
            )

spacing_df = pd.DataFrame(spacing_rows).sort_values(["dataset", "task", "split"], kind="stable")
display(spacing_df)

spacing_violations = int(spacing_df["violations"].sum()) if not spacing_df.empty else 0
display(Markdown(f"Spacing violations total: **{spacing_violations}**"))

In [ ]:
leakage_rows = []

for dataset_name, bundle in analysis.items():
    if bundle["dataset_error"] is not None:
        continue
    ds = bundle["dataset"]
    for task_name in bundle["task_names"]:
        task_obj = bundle["task_objs"][task_name]

        train_chk = leakage_check_against_boundary(task_name, task_obj, "train", ds.val_timestamp)
        leakage_rows.append(
            {
                "dataset": dataset_name,
                "task": task_name,
                "check": "train_vs_val_boundary",
                "boundary_ts": ds.val_timestamp,
                **train_chk,
            }
        )

        val_chk = leakage_check_against_boundary(task_name, task_obj, "val", ds.test_timestamp)
        leakage_rows.append(
            {
                "dataset": dataset_name,
                "task": task_name,
                "check": "val_vs_test_boundary",
                "boundary_ts": ds.test_timestamp,
                **val_chk,
            }
        )

leakage_df = pd.DataFrame(leakage_rows).sort_values(["dataset", "task", "check"], kind="stable")
display(leakage_df)

total_cross = int(leakage_df["cross_boundary_rows"].sum()) if not leakage_df.empty else 0
display(Markdown(f"Boundary-crossing rows total: **{total_cross}**"))

In [ ]:
strict_leakage_ok = True
if not spacing_df.empty and int(spacing_df["violations"].sum()) > 0:
    strict_leakage_ok = False
if not leakage_df.empty and int(leakage_df["cross_boundary_rows"].sum()) > 0:
    strict_leakage_ok = False

display(Markdown(f"Strict leakage audit passed: **{strict_leakage_ok}**"))
if not strict_leakage_ok:
    display(Markdown("Investigate rows in `spacing_df` and `leakage_df` before using those task/split settings."))

## 4) Train-Target Diagnostics

In [ ]:
target_rows = []
for dataset_name, bundle in analysis.items():
    if bundle["dataset_error"] is not None:
        continue
    for task_name in bundle["task_names"]:
        task_obj = bundle["task_objs"][task_name]
        stats = summarize_target(task_obj, split="train")
        target_rows.append(
            {
                "dataset": dataset_name,
                "task": task_name,
                "task_type": str(task_obj.task_type).split(".")[-1],
                "timedelta_hours": _safe_timedelta_hours(getattr(task_obj, "timedelta", None)),
                **stats,
            }
        )
target_df = pd.DataFrame(target_rows).sort_values(["dataset", "task"], kind="stable")
display(target_df)

## 5) Empirical Delta Motivation from Event-Object Recency

In [ ]:
timing_rows = []
for dataset_name, bundle in analysis.items():
    if bundle["dataset_error"] is not None:
        continue
    recent = event_object_recent_matrix(bundle["db"])
    if recent.empty:
        continue
    by_obj = (
        recent.groupby("object_type", observed=False)
        .agg(
            n=("n", "sum"),
            prev_delta_p50_hours=("prev_delta_p50_hours", "median"),
            prev_delta_p90_hours=("prev_delta_p90_hours", "median"),
        )
        .reset_index()
        .sort_values("n", ascending=False, kind="stable")
    )
    by_obj.insert(0, "dataset", dataset_name)
    timing_rows.append(by_obj)
timing_df = pd.concat(timing_rows, ignore_index=True) if timing_rows else pd.DataFrame()
display(timing_df)

## 6) Window-Size Growth by Task Deltas

In [ ]:
window_summary_rows = []
for dataset_name, bundle in analysis.items():
    if bundle["dataset_error"] is not None:
        continue
    deltas = sorted(
        {
            pd.Timedelta(getattr(task_obj, "timedelta"))
            for task_obj in bundle["task_objs"].values()
            if getattr(task_obj, "timedelta", None) is not None
        }
    )
    if not deltas:
        continue
    samples = sample_window_graph_sizes(
        bundle["db"],
        deltas,
        num_roots=WINDOW_GRAPH_NUM_ROOTS,
        random_state=WINDOW_GRAPH_RANDOM_STATE,
    )
    summary = summarize_sampled_window_graph_sizes(samples)
    summary.insert(0, "dataset", dataset_name)
    window_summary_rows.append(summary)

window_df = pd.concat(window_summary_rows, ignore_index=True) if window_summary_rows else pd.DataFrame()
display(window_df)

if not window_df.empty:
    plot_dataset = window_df["dataset"].iloc[0]
    plot_df = window_df[window_df["dataset"] == plot_dataset].copy()
    fig, ax = plt.subplots(figsize=(8, 4))
    for metric, marker in [("event", "o"), ("object", "s")]:
        cur = plot_df[plot_df["metric"] == metric].sort_values("delta_days", kind="stable")
        ax.plot(cur["delta_days"], cur["mean"], marker=marker, label=f"{metric}_count")
        ax.fill_between(cur["delta_days"], cur["ci_lower"], cur["ci_upper"], alpha=0.15)
    ax.set_title(f"Window Graph Growth by Delta ({plot_dataset})")
    ax.set_xlabel("Delta (days)")
    ax.set_ylabel("Estimated count")
    ax.legend()
    plt.show()

## 7) Weekly Activity Stability

In [ ]:
stability_rows = []
for dataset_name, bundle in analysis.items():
    if bundle["dataset_error"] is not None:
        continue
    event_df = bundle["db"].table_dict["event"].df.copy()
    event_df[TIME_COL] = pd.to_datetime(event_df[TIME_COL], errors="coerce")
    event_df = event_df.dropna(subset=[TIME_COL])
    for h in STABILITY_BUCKET_HOURS:
        metrics = summarize_interval_stability(event_df, hours=h, min_events=200)
        stability_rows.append({"dataset": dataset_name, **metrics})

stability_df = pd.DataFrame(stability_rows).sort_values(["dataset", "interval_hours"], kind="stable")
display(stability_df)

## 8) Consolidated View

In [ ]:
final_df = (
    catalog_df.merge(
        target_df[["dataset", "task", "positive_rate", "p50_hours", "p90_hours", "mean_hours"]],
        on=["dataset", "task"],
        how="left",
    )
    .merge(
        splits_df[["dataset", "task", "train_rows", "val_rows", "test_rows", "train_unique_ts", "val_unique_ts", "test_unique_ts"]],
        on=["dataset", "task"],
        how="left",
    )
    .sort_values(["dataset", "task"], kind="stable")
)
display(final_df)

if not leakage_df.empty:
    leak_summary = leakage_df.groupby(["dataset", "task"], observed=False)["cross_boundary_rows"].sum().reset_index()
    leak_summary = leak_summary.rename(columns={"cross_boundary_rows": "total_cross_boundary_rows"})
    display(leak_summary.sort_values(["dataset", "task"], kind="stable"))

display(Markdown("If `total_cross_boundary_rows == 0` and spacing violations are zero, the split policy is leakage-safe under this notebook's horizon definition."))